# 184 — Residual Stream Direction Analysis

Analyze important directions: unembed-aligned components,
maximally active dimensions, contribution tracking, diversity,
and cross-layer direction overlap.

## Why This Matters

Residual stream stream direction characterizes the shared information bus that all transformer components read from and write to. Because the residual stream carries all inter-component communication, understanding its structure is fundamental to understanding the model as a whole.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.residual_stream_direction_analysis import (
    unembed_aligned_directions,
    maximally_active_dimensions,
    direction_contribution_tracking,
    residual_direction_diversity,
    important_direction_overlap,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
result = unembed_aligned_directions(model, tokens, layer=2, top_k=5)
for d in result['directions']:
    promoted = ', '.join(f'tok{t["token"]}' for t in d['top_promoted'])
    print(f"PC{d['component']}: sv={d['singular_value']:.4f}  "
          f"var={d['variance_explained']:.3f}  promotes=[{promoted}]")

In [ ]:
result = maximally_active_dimensions(model, tokens, layer=2, top_k=5)
print(f"Top-5 account for {result['top_k_variance_share']:.1%} of variance\n")
for d in result['top_by_variance']:
    print(f"  Dim {d['dimension']:2d}: var={d['variance']:.4f}  "
          f"abs={d['mean_abs']:.4f}  top_logit_tok={d['top_logit_token']}")

In [ ]:
direction = jax.random.normal(jax.random.PRNGKey(0), (32,))
result = direction_contribution_tracking(model, tokens, direction)
print(f"Cumulative contribution: {result['cumulative']:.4f}\n")
for layer in result['layers']:
    heads = ', '.join(f'H{h["head"]}={h["projection"]:.3f}' for h in layer['per_head'])
    print(f"Layer {layer['layer']}: attn={layer['attn_contribution']:.4f}  "
          f"mlp={layer['mlp_contribution']:.4f}  [{heads}]")

In [ ]:
result = residual_direction_diversity(model, tokens)
for layer in result['per_layer']:
    bar = '#' * int(layer['direction_diversity'] * 30)
    print(f"Layer {layer['layer']}: diversity={layer['direction_diversity']:.3f}  "
          f"eff_dim={layer['effective_dimensionality']:.1f}  {bar}")

In [ ]:
result = important_direction_overlap(model, tokens)
print(f"Mean overlap: {result['mean_overlap']:.3f}\n")
for t in result['transitions']:
    print(f"L{t['from_layer']}->L{t['to_layer']}: mean={t['mean_overlap']:.3f}  "
          f"range=[{t['min_overlap']:.3f}, {t['max_overlap']:.3f}]")